In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv(
    "btc_usdt_2h_raw.csv",
    parse_dates=["timestamp"],
    index_col="timestamp"
)

In [ ]:
df_feat = df.copy()

In [ ]:
# Testing Features:

df_feat['return_1'] = df_feat['close'].pct_change()
df_feat['vol_10'] = df_feat['return_1'].rolling(10).std()
df_feat['mom_10'] = (df_feat['close'] / df_feat['close'].rolling(10).mean() - 1)
df_feat['return_5'] = df_feat['close'].pct_change(5)
df_feat['ret_vol_ratio'] = df_feat['return_1'] / (df_feat['vol_10'] + 1e-6)
df_feat['vol_30'] = df_feat['return_1'].rolling(30).std()
df_feat['trend_strenght'] = df_feat['mom_10'] / (df_feat['vol_10'] + 1e-6)

In [ ]:
# Defining y
df_feat['target'] = (df_feat['return_1'].shift(-1) > 0).astype(int)

In [ ]:
df_feat = df_feat.dropna()

In [ ]:
n = len(df_feat)

train_end = int(n * 0.6)
val_end   = int(n * 0.8)

df_train = df_feat.iloc[:train_end]
df_val   = df_feat.iloc[train_end:val_end]
df_test  = df_feat.iloc[val_end:]

In [ ]:
def evaluate(features):
    X_train = df_train[features].values
    y_train = df_train['target'].values

    X_val = df_val[features].values
    y_val = df_val['target'].values

    X_test = df_test[features].values
    y_test = df_test['target'].values

    
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    val_pred = model.predict_proba(X_val)[:,1]
    test_pred = model.predict_proba(X_test)[:,1]

    val_auc = roc_auc_score(y_val, val_pred)
    test_auc = roc_auc_score(y_test, test_pred)

    return val_auc, test_auc

In [ ]:
# Feature 1

evaluate(['return_1'])

In [ ]:
# Feature 2

evaluate(['return_1', 'vol_10'])

# Here feature 2 doesn't genralize.

In [ ]:
# Feature 3

evaluate(['return_1', 'mom_10'])

In [ ]:
# Feature 4

evaluate(['return_1', 'mom_10', 'return_5'])

# Here it also doresn't genralize

In [ ]:
# Feature 5

evaluate(['return_1', 'mom_10', 'ret_vol_ratio'])

# Here feature 5 also doesnt genralise

In [ ]:
# Feature 6

evaluate(['return_1', 'mom_10', 'vol_30'])

In [ ]:
# Feature 7

evaluate(['return_1', 'mom_10', 'vol_30', 'trend_strenght'])

# Here it also doesnt genralize

In [ ]:
# Final Features Model LR:

evaluate(['return_1', 'mom_10', 'vol_30'])

In [ ]:
# Final Model

features = ['return_1', 'mom_10', 'vol_30']

X_train = df_train[features].values
y_train = df_train['target'].values

X_val = df_val[features].values
y_val = df_val['target'].values

X_test = df_test[features].values
y_test = df_test['target'].values

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

val_pred = model.predict_proba(X_val)[:,1]
test_pred = model.predict_proba(X_test)[:,1]

val_auc = roc_auc_score(y_val, val_pred)
test_auc = roc_auc_score(y_test, test_pred)

val_auc, test_auc

#### Error Analysis

In [ ]:
df_val_analysis = df_val.copy()

df_val_analysis['pred_prob'] = val_pred
df_val_analysis['pred_label'] = (val_pred > 0.5).astype(int)
df_val_analysis['error'] = (
    df_val_analysis['pred_label'] != df_val_analysis['target']
)

In [ ]:
# High Confidence Errors:

df_val_analysis[
    df_val_analysis['error']
].sort_values('pred_prob', ascending=False).head(10)

The model fails when volatility is high AND momentum is weak/negative.

#### Skipping simulating trades in bad rgimes:

In [ ]:
vol_threshold = df_val_analysis['vol_30'].quantile(0.8)
vol_threshold

In [ ]:
bad_regime = (
    (df_val_analysis['vol_30'] > vol_threshold) &
    (df_val_analysis['mom_10'] < 0)
)

In [ ]:
# Baseline Error
df_val_analysis['error'].mean()

In [ ]:
# Filtered Error 
df_val_analysis.loc[~bad_regime, 'error'].mean()

By Applying filter regime on high volatility and weak momentum we decreased error rate slightly and model must avoid to trade these regimes.

In [ ]:
trades_skipped = (bad_regime).sum()/ len(df_val_analysis)*100
trades_skipped

Here we have skipped ~10 percent of of all trades.

### Backtesting

In [ ]:
df_bt = df_test.copy()

df_bt['future_return'] = df_bt['close'].pct_change().shift(-1)


In [ ]:
df_bt['pred_prob'] = test_pred

In [ ]:
trade_threshold = 0.55

bad_regime_test = (
    (df_bt['vol_30'] > vol_threshold) &
    (df_bt['mom_10'] < 0)
)

df_bt['trade'] = (
    (df_bt['pred_prob'] > trade_threshold)
    & (~bad_regime_test)   
)


In [ ]:
df_bt['strategy_return'] = df_bt['trade'] * df_bt['future_return']

In [ ]:
df_bt['equity'] = (1 + df_bt['strategy_return']).cumprod()


In [ ]:
df_bt = df_bt.dropna()

In [ ]:
total_return = df_bt['equity'].iloc[-1] - 1
num_trades = df_bt['trade'].sum()
avg_trade_return = df_bt.loc[df_bt['trade'], 'future_return'].mean()
max_drawdown = (
    df_bt['equity'] / df_bt['equity'].cummax() - 1
).min()


In [ ]:
total_return, num_trades, avg_trade_return, max_drawdown 

In [ ]:
equity_plot = plt.plot(df_bt['equity'])
equity_plot